# Module 3.2: Transformer 编码器 (Transformer Encoder)

## 1. 本章概览

### 📚 学习目标

1. **编码器组件**：理解前馈网络、层归一化、残差连接
2. **编码器层**：实现完整的单层编码器
3. **编码器堆栈**：构建多层编码器架构
4. **实践应用**：在真实任务上应用编码器

### 🎯 核心问题

- Transformer 编码器由哪些组件构成？
- 为什么需要残差连接和层归一化？
- 如何堆叠多个编码器层？

### 🗺️ 知识地图

```
Transformer 架构
├── 编码器 (Encoder) ← 本章重点
│   ├── 多头自注意力
│   ├── 前馈网络 (FFN)
│   ├── 层归一化 (LayerNorm)
│   └── 残差连接 (Residual)
└── 解码器 (Decoder)
```

### ⏱️ 预计学习时间：4-5小时

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

print("✓ Libraries imported")

## 2. 位置前馈网络 (Position-wise Feed-Forward Network)

### 2.1 什么是 FFN？

在 Transformer 中，每个编码器层包含一个**位置前馈网络**（FFN），它独立地应用于序列的每个位置。

### 2.2 数学公式

$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

**参数**：
- $W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$: 第一层权重（扩展维度）
- $b_1 \in \mathbb{R}^{d_{ff}}$: 第一层偏置
- $W_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$: 第二层权重（压缩维度）
- $b_2 \in \mathbb{R}^{d_{model}}$: 第二层偏置

**典型维度**：
- $d_{model} = 512$（模型维度）
- $d_{ff} = 2048$（前馈网络内部维度，通常是 $4 \times d_{model}$）

In [ ]:
# 🔬 Micro Practice: Position-wise Feed-Forward Network
# Goal: Implement FFN from scratch

class PositionwiseFeedForward:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)
        self.b2 = np.zeros(d_model)
    
    def forward(self, x):
        hidden = np.maximum(0, x @ self.W1 + self.b1)
        output = hidden @ self.W2 + self.b2
        return output

# Test
d_model, d_ff, seq_len = 8, 32, 5
x = np.random.randn(seq_len, d_model)
ffn = PositionwiseFeedForward(d_model, d_ff)
output = ffn.forward(x)

print(f"Input: {x.shape}, Output: {output.shape}")
print("✓ FFN implemented!")

## 3. 层归一化 (Layer Normalization)

### 3.1 为什么需要归一化？

归一化技术可以：稳定训练、加速收敛、减少对初始化的敏感性

### 3.2 Layer Normalization 公式

$$\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

其中：
- $\mu = \frac{1}{d}\sum_{i=1}^{d} x_i$：特征维度的均值
- $\sigma^2 = \frac{1}{d}\sum_{i=1}^{d} (x_i - \mu)^2$：特征维度的方差
- $\gamma, \beta$：可学习的缩放和平移参数

In [ ]:
# 🔬 Micro Practice: Layer Normalization

class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps
    
    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        std = x.std(axis=-1, keepdims=True)
        normalized = (x - mean) / (std + self.eps)
        return self.gamma * normalized + self.beta

# Test
ln = LayerNorm(d_model=8)
x = np.random.randn(5, 8)
output = ln.forward(x)

print(f"Input mean: {x.mean(axis=-1)}")
print(f"Output mean: {output.mean(axis=-1)}")
print(f"Output std: {output.std(axis=-1)}")
print("✓ LayerNorm implemented!")

## 4. 残差连接和 Dropout

### 4.1 残差连接 (Residual Connection)

残差连接允许梯度直接流过网络，解决深层网络的梯度消失问题：

$$\text{output} = \text{LayerNorm}(x + \text{Sublayer}(x))$$

### 4.2 Dropout

Dropout 用于正则化，在训练时随机丢弃一些神经元：

- 应用位置：注意力输出后、FFN 输出后、嵌入层
- 典型值：0.1

In [ ]:
# 🔬 Micro Practice: Residual Connection + Dropout

def sublayer_connection(x, sublayer, layer_norm, dropout_rate=0.1, training=True):
    """
    Apply: LayerNorm(x + Dropout(Sublayer(x)))
    """
    # Apply sublayer
    sublayer_output = sublayer(x)
    
    # Apply dropout (only during training)
    if training and dropout_rate > 0:
        mask = np.random.binomial(1, 1-dropout_rate, sublayer_output.shape)
        sublayer_output = sublayer_output * mask / (1 - dropout_rate)
    
    # Residual connection
    output = x + sublayer_output
    
    # Layer normalization
    output = layer_norm.forward(output)
    
    return output

# Test
x = np.random.randn(5, 8)
ffn = PositionwiseFeedForward(8, 32)
ln = LayerNorm(8)
output = sublayer_connection(x, ffn.forward, ln, dropout_rate=0.1)

print(f"Input: {x.shape}, Output: {output.shape}")
print("✓ Residual connection implemented!")

## 5. 单个编码器层 (Single Encoder Layer)

### 5.1 编码器层架构

一个完整的编码器层包含两个子层：

1. **多头自注意力** (Multi-Head Self-Attention)
2. **位置前馈网络** (Position-wise FFN)

每个子层都使用：
- 残差连接 (Residual Connection)
- 层归一化 (Layer Normalization)
- Dropout（可选）

### 5.2 数据流

```
输入 x (seq_len, d_model)
  ↓
多头自注意力
  ↓
Add & Norm (x + Attention(x))
  ↓
前馈网络 (FFN)
  ↓
Add & Norm (x + FFN(x))
  ↓
输出 (seq_len, d_model)
```

In [ ]:
# 🔬 Micro Practice: Single Encoder Layer

# Simplified attention for demonstration
def simple_self_attention(x):
    """Simplified self-attention (placeholder)"""
    # In practice, use scaled dot-product attention from previous notebook
    d_k = x.shape[-1]
    scores = x @ x.T / np.sqrt(d_k)
    attn_weights = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)
    return attn_weights @ x

class EncoderLayer:
    def __init__(self, d_model, d_ff, dropout=0.1):
        self.ffn = PositionwiseFeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.dropout = dropout
    
    def forward(self, x, training=True):
        # Sub-layer 1: Self-Attention + Add & Norm
        attn_output = simple_self_attention(x)
        if training and self.dropout > 0:
            mask = np.random.binomial(1, 1-self.dropout, attn_output.shape)
            attn_output = attn_output * mask / (1 - self.dropout)
        x = self.norm1.forward(x + attn_output)
        
        # Sub-layer 2: FFN + Add & Norm
        ffn_output = self.ffn.forward(x)
        if training and self.dropout > 0:
            mask = np.random.binomial(1, 1-self.dropout, ffn_output.shape)
            ffn_output = ffn_output * mask / (1 - self.dropout)
        x = self.norm2.forward(x + ffn_output)
        
        return x

# Test
encoder_layer = EncoderLayer(d_model=8, d_ff=32, dropout=0.1)
x = np.random.randn(5, 8)
output = encoder_layer.forward(x, training=False)

print(f"Input: {x.shape}, Output: {output.shape}")
print("✓ Encoder layer implemented!")

## 6. 完整编码器堆栈 (Full Encoder Stack)

### 6.1 堆叠多层

Transformer 编码器通常堆叠 N 个相同的编码器层：

- **原始 Transformer**: N = 6
- **BERT-Base**: N = 12
- **BERT-Large**: N = 24
- **GPT-3**: N = 96

### 6.2 为什么堆叠？

- **层次化表示**：浅层捕获局部特征，深层捕获全局语义
- **增加容量**：更多参数，更强的表达能力
- **渐进式抽象**：逐层提炼信息

In [ ]:
# 🔬 Micro Practice: Full Encoder Stack

class TransformerEncoder:
    def __init__(self, num_layers, d_model, d_ff, dropout=0.1):
        self.layers = [EncoderLayer(d_model, d_ff, dropout) 
                       for _ in range(num_layers)]
        self.num_layers = num_layers
    
    def forward(self, x, training=True):
        # Pass through each encoder layer
        for i, layer in enumerate(self.layers):
            x = layer.forward(x, training)
        return x

# Test
encoder = TransformerEncoder(num_layers=3, d_model=8, d_ff=32, dropout=0.1)
x = np.random.randn(5, 8)
output = encoder.forward(x, training=False)

print(f"Input: {x.shape}")
print(f"Output after {encoder.num_layers} layers: {output.shape}")
print("✓ Full encoder stack implemented!")

## 7. PyTorch 实现

### 7.1 为什么使用 PyTorch？

- **自动微分**：自动计算梯度
- **GPU 加速**：高效的矩阵运算
- **模块化**：nn.Module 提供清晰的结构
- **生态系统**：丰富的预训练模型和工具

In [ ]:
# 🔬 Micro Practice: PyTorch Transformer Encoder

class PyTorchEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention
        attn_output, _ = self.self_attn(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # FFN
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        
        return x

class PyTorchTransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, nhead, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            PyTorchEncoderLayer(d_model, nhead, d_ff, dropout)
            for _ in range(num_layers)
        ])
    
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x

# Test
batch_size, seq_len, d_model = 2, 5, 64
x = torch.randn(batch_size, seq_len, d_model)

encoder = PyTorchTransformerEncoder(num_layers=3, d_model=64, nhead=8, d_ff=256, dropout=0.1)
output = encoder(x)

print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Parameters: {sum(p.numel() for p in encoder.parameters()):,}")
print("✓ PyTorch encoder implemented!")

## 8. 总结与展望

### 核心要点

1. **FFN**：位置独立的前馈网络，先扩展后压缩
2. **LayerNorm**：在特征维度归一化，稳定训练
3. **残差连接**：解决梯度消失，允许深层网络
4. **编码器层**：自注意力 + FFN，各带残差和归一化
5. **编码器堆栈**：多层堆叠，层次化表示

### 常见问题

**Q1: Pre-Norm vs Post-Norm？**
- Pre-Norm：更稳定，更容易训练深层网络
- Post-Norm：原始 Transformer 使用，但训练较困难

**Q2: 需要多少层？**
- 小任务：3-6 层
- 中等任务：6-12 层（BERT-Base）
- 大型任务：12-24 层（BERT-Large）

**Q3: 计算复杂度？**
- 自注意力：$O(n^2 \cdot d)$
- FFN：$O(n \cdot d^2)$
- 总体：$O(n^2 \cdot d + n \cdot d^2)$

### 下一章预告

**Module 3.3: Transformer Decoder**
- 解码器架构
- 交叉注意力
- 自回归生成

### 💡 思考题

1. 为什么 FFN 的隐藏层维度通常是 4×d_model？
2. 如果去掉残差连接会发生什么？
3. 如何修改编码器来处理图结构数据？
4. Pre-Norm 和 Post-Norm 在数学上有什么区别？

## 思考题参考答案

### 问题 1：为什么 FFN 的隐藏层维度通常是 4×d_model？

**答案：**

原始 Transformer 论文（Vaswani et al., 2017）设定 $d_{ff} = 4 \times d_{model}$（即 $512 \times 4 = 2048$），这一设计背后有多重考量：

**理论解释：**

1. **表达能力与瓶颈设计**：FFN 形成一个"先扩展再压缩"的瓶颈结构。扩展阶段（$d_{model} \to d_{ff}$）让网络在高维空间中学习复杂的非线性特征；压缩阶段（$d_{ff} \to d_{model}$）筛选出对任务最有用的信息。这类似于自编码器的思想。

2. **补偿注意力的局限**：自注意力本质上是线性操作（对 Value 的加权求和），缺乏非线性变换能力。FFN 用 ReLU 等激活函数提供必要的非线性，更宽的 FFN 可以弥补这一不足。

3. **参数分配均衡**：在标准 Transformer 中，每层的参数量分布约为：
   - 自注意力（含 $W^Q, W^K, W^V, W^O$）：$4 \times d_{model}^2$
   - FFN（含 $W_1, W_2$）：$d_{model} \times d_{ff} + d_{ff} \times d_{model} = 2 \times d_{model} \times d_{ff}$

   当 $d_{ff} = 4 \times d_{model}$ 时，FFN 参数量为 $8 \times d_{model}^2$，约是注意力参数量的 2 倍。这使 FFN 承担了模型中更多的"存储"功能（研究表明 FFN 可以视为键值存储，记忆事实知识）。

4. **经验验证**：大量实验表明 4 倍是效率与性能的良好平衡点。比例过小（如 1×）导致欠拟合；比例过大（如 8×）提升有限但计算成本翻倍。

**现代变体：**

| 模型 | $d_{model}$ | $d_{ff}$ | 比例 | 激活函数 |
|------|------------|---------|------|---------|
| BERT-Base | 768 | 3072 | 4× | GELU |
| GPT-2 | 768 | 3072 | 4× | GELU |
| LLaMA-2 7B | 4096 | 11008 | ~2.7× | SwiGLU |
| GPT-3 175B | 12288 | 49152 | 4× | GeLU |

注意：使用 SwiGLU 等门控激活函数的模型通常将比例调整为约 $\frac{8}{3} \times$ 以保持参数量不变（因为门控机制引入了额外参数）。

---

### 问题 2：如果去掉残差连接会发生什么？

**答案：**

残差连接（Residual Connection）是 Transformer 能够训练深层网络的关键设计。去掉后会产生以下严重问题：

**问题一：梯度消失（最核心的问题）**

在没有残差连接的情况下，反向传播时梯度需要连乘经过每一层的雅可比矩阵：

$$\frac{\partial L}{\partial x_0} = \frac{\partial L}{\partial x_N} \cdot \prod_{i=1}^{N} \frac{\partial x_i}{\partial x_{i-1}}$$

如果每层的雅可比矩阵的谱范数小于 1，梯度指数级衰减；若大于 1，则梯度爆炸。对于 6 层以上的网络，这个问题会变得非常严重。

**有残差连接时**，梯度可以沿"高速公路"直接流回早期层：

$$\frac{\partial L}{\partial x_i} = \frac{\partial L}{\partial x_{i+1}} \cdot \left(1 + \frac{\partial F(x_i)}{\partial x_i}\right)$$

其中常数项 "1" 确保了即使 $F(x_i)$ 的梯度很小，梯度也能正常传播。

**问题二：表示退化**

没有残差连接时，每一层必须从头开始学习变换。深层网络容易出现"表示坍塌"——不同位置的隐藏状态变得相似，丧失区分能力。

**问题三：恒等映射难以学习**

当某一层不需要做任何变换时，残差连接允许该层通过将权重置零来实现恒等映射，学习难度极低。没有残差连接时，层需要显式地学习恒等变换（难度大）。

**实验证据：**

He et al.（2016，ResNet 论文）证明，在图像任务中，没有残差连接的 56 层网络比 20 层网络表现更差，说明深层网络若无残差会产生退化问题。Transformer 中结论类似。

**总结：**

| 有无残差连接 | 梯度流动 | 最大可训练层数 | 收敛速度 |
|------------|---------|-------------|---------|
| 有 | 畅通 | 数百层（GPT-3: 96层） | 快 |
| 无 | 指数衰减/爆炸 | 约 6-8 层 | 慢或发散 |

---

### 问题 3：如何修改编码器来处理图结构数据？

**答案：**

标准 Transformer 编码器处理的是序列（1D 结构），将其扩展到图结构数据（Graph）需要对注意力机制和位置编码进行针对性修改。

**核心挑战：**

- 图中节点没有固定的线性顺序
- 节点之间的连接关系（边）需要被显式建模
- 图可能有不同的拓扑结构

**方案一：图注意力（将注意力限制在邻域内）**

标准注意力：每个节点关注所有节点

图注意力（Graph Attention Network, GAT）：每个节点只关注其邻居：

$$h_i' = \sum_{j \in \mathcal{N}(i)} \alpha_{ij} W h_j$$

$$\alpha_{ij} = \text{softmax}_j(e_{ij}), \quad e_{ij} = \text{LeakyReLU}(a^T [W h_i \| W h_j])$$

**方案二：加入边特征（Edge-aware Attention）**

在计算注意力分数时融入边类型信息：

$$\text{score}(i, j) = \frac{(h_i W^Q)(h_j W^K + e_{ij} W^E)^T}{\sqrt{d_k}}$$

其中 $e_{ij}$ 是边 $i \to j$ 的特征嵌入。代表工作：Graphformer、GRPE。

**方案三：修改位置编码为图位置编码**

将序列位置编码替换为能反映图拓扑的编码：

- **最短路径编码**：用节点对之间的最短路径距离作为位置偏置
- **随机游走编码**（RWPE）：用从节点出发的随机游走概率分布编码结构信息
- **拉普拉斯特征向量**（LapPE）：图拉普拉斯矩阵的特征向量蕴含全局拓扑信息

**方案四：Graphormer（微软研究院）**

结合上述思想，在注意力分数中加入以下偏置：

$$\text{score}(i,j) = \frac{h_i W^Q (h_j W^K)^T}{\sqrt{d_k}} + b_{\phi(i,j)} + c_{e_{ij}}$$

- $b_{\phi(i,j)}$：基于最短路径距离的标量偏置
- $c_{e_{ij}}$：基于边特征的偏置

**方案五：将图转化为序列**

使用图的 BFS/DFS 遍历顺序或 Weisfeiler-Leman 图同构测试的结果，将图节点序列化后输入标准 Transformer。

**应用场景：**

| 应用 | 图类型 | 代表方法 |
|------|--------|---------|
| 分子性质预测 | 分子图 | Graphormer |
| 知识图谱推理 | 知识图 | KGPT |
| 社交网络分析 | 异构图 | HGT |
| 代码理解 | AST/CFG | Code2Vec + GNN |

---

### 问题 4：Pre-Norm 和 Post-Norm 在数学上有什么区别？

**答案：**

**Post-Norm（原始 Transformer）：**

$$x_{l+1} = \text{LayerNorm}(x_l + F_l(x_l))$$

归一化在残差连接之后进行。

**Pre-Norm（现代模型常用）：**

$$x_{l+1} = x_l + F_l(\text{LayerNorm}(x_l))$$

归一化在子层之前进行（作用于输入）。

**数学分析：**

**梯度传播对比：**

Post-Norm 的梯度（从第 $L$ 层到第 $l$ 层）：

$$\frac{\partial x_L}{\partial x_l} = \prod_{k=l}^{L-1} \frac{\partial \text{LN}(x_k + F_k(x_k))}{\partial (x_k + F_k(x_k))} \cdot \left(I + \frac{\partial F_k(x_k)}{\partial x_k}\right)$$

LayerNorm 的雅可比矩阵可能接近于零（当输入很小时），导致梯度消失。

Pre-Norm 的梯度：

$$\frac{\partial x_L}{\partial x_l} = \prod_{k=l}^{L-1} \left(I + \frac{\partial F_k(\text{LN}(x_k))}{\partial x_k}\right)$$

残差路径的雅可比矩阵中有单位矩阵 $I$，保证梯度至少为 1，不会消失。

**表示尺度的差异：**

- **Post-Norm**：每层输出经过归一化，层间信号尺度受控，但早期层梯度小
- **Pre-Norm**：残差流（主干）的 $x_l$ 不经过归一化，随深度增加幅值可能增大（即"表示缩放"现象）

研究（Xiong et al. 2020）表明，Pre-Norm 在初始化时的梯度尺度更均匀，这是它更容易训练的根本原因。

**训练稳定性对比：**

| 特性 | Post-Norm | Pre-Norm |
|------|-----------|---------|
| 梯度流动 | 不稳定，需要 warmup | 稳定，可以更大学习率 |
| 初始化敏感性 | 高（需要精心设计） | 低 |
| 最终性能 | 通常更高（充分训练后） | 略低但可接受 |
| 训练难度 | 较难（深层网络） | 较易 |
| 是否需要 warmup | 是 | 不一定 |

**DeepNorm（微软 2022）：**

为了兼得两者优点，提出 DeepNorm：

$$x_{l+1} = \text{LayerNorm}(\alpha \cdot x_l + F_l(x_l))$$

其中 $\alpha > 1$ 是一个缩放系数，同时配合特定的初始化策略，使得可以稳定训练 1000 层以上的 Transformer。

**实践建议：**

- 训练中小型模型：两者差异不大，Post-Norm 最终性能可能更好
- 训练超深模型（>24 层）：强烈推荐 Pre-Norm 或 DeepNorm
- 需要快速实验：使用 Pre-Norm，减少调参成本
